In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve,confusion_matrix
from sklearn.preprocessing import StandardScaler
import sys
sys.path.append("eval.py") 
from eval import notation


In [33]:
# importation des données

data_1 = pd.read_csv("./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
data_2 = pd.read_csv("./data/2023-02-12.csv")

data_1.columns = data_1.columns.str.replace(" ", "").str.lower()
data_2.columns = data_2.columns.str.replace(" ", "").str.lower()

# on garde que les colonnes communes
common_columns = list(set(data_1.columns) & set(data_2.columns))

data_1 = data_1[common_columns]
data_2 = data_2[common_columns]

# on concatène les deux datasets
concatenated_data = pd.concat([data_1, data_2], ignore_index=True)
concatenated_data["label_ml"] = (concatenated_data["label"] != "BENIGN").astype(int)
concatenated_data["label_dl"]  = concatenated_data["label"]
concatenated_data = concatenated_data.drop(columns=["label"])

concatenated_data.replace([np.inf, -np.inf], np.nan, inplace=True)
concatenated_data.dropna(inplace=True)
concatenated_data = concatenated_data.sort_values(by="label_dl")

label_mapping = {
    'DDoS': 'ddos',
    'ddospot': 'ddos',
    'cowrie': 'SSH',
    'log4pot': 'SSH'
}

# Appliquer le mappage aux labels
concatenated_data['label_dl'] = concatenated_data['label_dl'].map(label_mapping).fillna(concatenated_data['label_dl'])

In [34]:
# préparation de la donnée pour la mettre dans le modèle de machine learning

X_ml = concatenated_data.drop(['label_dl', 'label_ml'], axis=1)
y_ml = concatenated_data['label_ml']

scaler = StandardScaler()
X_ml = scaler.fit_transform(X_ml)

X_train_ml, X_test_ml, y_train_ml, y_test_ml = train_test_split(X_ml, y_ml, test_size=0.2, random_state=42)


In [ ]:
# Determination des meilleurs paramètres grâce à la recherche randomisée

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer, confusion_matrix, recall_score
import numpy as np

model = DecisionTreeClassifier(random_state=42)

param_dist = {
    'max_depth': np.arange(1, 21),
    'min_samples_split': np.arange(2, 21),
    'min_samples_leaf': np.arange(1, 21),
    'criterion': ['gini', 'entropy']
}

# Définir une fonction de scoring personnalisée, dans notre domaine, on veut limiter le nombre de faux positifs

def custom_scorer(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return (tp - fp) / (tp + fp + 1e-6)

random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=100,
    scoring=make_scorer(custom_scorer),
    cv=5,
    random_state=42,
    n_jobs=-1  
)

random_search.fit(X_train_ml, y_train_ml)

print("Meilleurs paramètres (Random Search):", random_search.best_params_)
print("Meilleur score (Random Search):", random_search.best_score_)


Meilleurs paramètres (Random Search): {'min_samples_split': np.int64(15), 'min_samples_leaf': np.int64(6), 'max_depth': np.int64(9), 'criterion': 'entropy'}
Meilleur score (Random Search): 0.9997060562874536


In [ ]:
import optuna
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, confusion_matrix


# Determination des meilleurs paramètres grâce à la recherche bayésienne

def objective(trial):
    max_depth = trial.suggest_int('max_depth', 1, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    criterion = trial.suggest_categorical('criterion', ['gini', 'entropy'])

    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        criterion=criterion,
        random_state=42
    )

    def custom_scorer(y_true, y_pred):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        return (tp - fp) / (tp + fp + 1e-6)  

    score = cross_val_score(model, X_train_ml, y_train_ml, cv=5, scoring=make_scorer(custom_scorer))

    return score.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Meilleurs paramètres (Optuna):", study.best_params)
print("Meilleur score (Optuna):", study.best_value)


c:\Users\allan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-02-25 11:22:13,545] A new study created in memory with name: no-name-ce235f4c-4c1b-455b-bade-05ee2134d3ff
[I 2025-02-25 11:22:22,501] Trial 0 finished with value: 0.8780528906590106 and parameters: {'max_depth': 3, 'min_samples_split': 20, 'min_samples_leaf': 8, 'criterion': 'gini'}. Best is trial 0 with value: 0.8780528906590106.
[I 2025-02-25 11:22:42,112] Trial 1 finished with value: 0.9996360679797427 and parameters: {'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 12, 'criterion': 'entropy'}. Best is trial 1 with value: 0.9996360679797427.
[I 2025-02-25 11:22:51,481] Trial 2 finished with value: 0.9487463685653076 and parameters: {'max_depth': 4, 'min_samples_split': 20, 'min_samples_leaf

Meilleurs paramètres (Optuna): {'max_depth': 8, 'min_samples_split': 9, 'min_samples_leaf': 8, 'criterion': 'entropy'}
Meilleur score (Optuna): 0.9997480236141143


In [35]:
# selection des meilleurs paramètres trouvés par les recherches
# 'max_depth': 8, 'min_samples_split': 9, 'min_samples_leaf': 8, 'criterion': 'entropy'

model_ml = DecisionTreeClassifier(max_depth=8, min_samples_split=9 , min_samples_leaf=8, criterion='entropy')
model_ml.fit(X_train_ml, y_train_ml)

concatenated_data["predict_ml"] = model_ml.predict(X_ml)

In [36]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from collections import Counter
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Entraînement du modèle de deep learning avec les données malwares
data_dl = concatenated_data[concatenated_data["label_ml"] != 0]

X_dl = data_dl.drop(['label_dl', 'label_ml', 'predict_ml'], axis=1)
y_dl = data_dl['label_dl']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_dl)

scaler = StandardScaler()
X_dl = scaler.fit_transform(X_dl)

initial_counts = Counter(y_encoded)
print(f"Distribution initiale des labels: {initial_counts}")

# Fixer un nombre cible pour chaque classe
majority_class = max(initial_counts, key=initial_counts.get)
target_size = int(initial_counts[majority_class] * 0.5)  # Réduire la classe majoritaire à 50%

# Stratégie SMOTE : Augmenter les classes minoritaires
smote_strategy = {k: max(v, target_size) for k, v in initial_counts.items() if v < target_size}

# Stratégie Under-sampling : Réduire la classe majoritaire
under_strategy = {majority_class: target_size}

# Pipeline avec SMOTE puis Under-sampling
resampling_pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=smote_strategy, random_state=42)),
    ('under', RandomUnderSampler(sampling_strategy=under_strategy, random_state=42))
])

# Appliquer le rééquilibrage
X_resampled, y_resampled = resampling_pipeline.fit_resample(X_dl, y_encoded)

# Vérifier la nouvelle distribution des labels
print(f"Nouvelle distribution des labels: {Counter(y_resampled)}")


Distribution initiale des labels: Counter({np.int64(3): 200291, np.int64(0): 3404, np.int64(1): 180, np.int64(2): 175, np.int64(4): 62, np.int64(5): 49, np.int64(6): 35})
Nouvelle distribution des labels: Counter({np.int64(0): 100145, np.int64(1): 100145, np.int64(2): 100145, np.int64(3): 100145, np.int64(4): 100145, np.int64(5): 100145, np.int64(6): 100145})


In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Convertir les labels encodés en format catégoriel
y_categorical = to_categorical(y_resampled)

# Diviser les données rééquilibrées en ensembles d'entraînement et de test
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_resampled, y_categorical, test_size=0.2, random_state=42)

# Construction du modèle ANN
model_dl = Sequential()
model_dl.add(Dense(64, input_dim=X_train_dl.shape[1], activation='relu'))
model_dl.add(Dense(32, activation='relu'))
model_dl.add(Dense(16, activation='relu'))
model_dl.add(Dense(y_categorical.shape[1], activation='softmax'))  # Pour une classification multi-classes

# Compilation du modèle
model_dl.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

# Entraînement du modèle
history = model_dl.fit(X_train_dl, y_train_dl, epochs=50, batch_size=32, validation_split=0.2)

# Évaluation du modèle
loss, accuracy = model_dl.evaluate(X_test_dl, y_test_dl)
print(f"Loss: {loss}")
print(f"Accuracy: {accuracy}")


c:\Users\allan\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 21s 1ms/step - accuracy: 0.7026 - loss: 0.7792 - val_accuracy: 0.7672 - val_loss: 0.5711
Epoch 2/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 18s 1ms/step - accuracy: 0.7777 - loss: 0.5661 - val_accuracy: 0.7912 - val_loss: 0.5255
Epoch 3/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 18s 1ms/step - accuracy: 0.7996 - loss: 0.5173 - val_accuracy: 0.8187 - val_loss: 0.5088
Epoch 4/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step - accuracy: 0.8137 - loss: 0.4821 - val_accuracy: 0.8177 - val_loss: 0.4574
Epoch 5/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 17s 1ms/step - accuracy: 0.8244 - loss: 0.4607 - val_accuracy: 0.8309 - val_loss: 0.4461
Epoch 6/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 19s 1ms/step - accuracy: 0.8306 - loss: 0.4455 - val_accuracy: 0.8375 - val_loss: 0.4233
Epoch 7/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 23s 2ms/step - accuracy: 0.8325 - loss: 0.4408 - val_accuracy: 0.7995 - val_loss: 0.4824
Epoch 8/50
14021/14021 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - accuracy: 

In [38]:
# reconstruction du dataframe avec les prédictions des deux modèles

data_dl = concatenated_data[concatenated_data["predict_ml"] == 1]

# On applique le réseau de neuronne sur les données prédite en malware par le decision tree
col_lab = data_dl[['label_dl', 'label_ml', 'predict_ml']]
data_final = data_dl.drop(['label_dl', 'label_ml', 'predict_ml'], axis=1)

y_pred_dl = model_dl.predict(data_final)

y_pred_labels = label_encoder.inverse_transform(np.argmax(y_pred_dl, axis=1))

data_final['predict_dl'] = y_pred_labels
data_final['label_dl'] = data_dl['label_dl']
data_final

6378/6378 ━━━━━━━━━━━━━━━━━━━━ 4s 636us/step


,fwdpacketlengthmax,bwdiatmin,subflowbwdbytes,fwdpackets/s,bwdiatstd,flowduration,pshflagcount,fwdiatstd,bwdpacketlengthmax,bwdpacketlengthmean,...,flowpackets/s,activemax,bwdpackets/s,subflowfwdbytes,fwdiatmax,finflagcount,bwdpacketlengthmin,bwdiatmean,predict_dl,label_dl
59480,6.0,0.0,0,6.041247,0.000000e+00,662115,0,3.473955e+05,0.0,0.000,...,6.041247,0.0,0.000000,24,621145.0,0,0.0,0.000000e+00,ddos,BENIGN
198021,6.0,10000000.0,12,0.197140,0.000000e+00,15217590,0,3.446834e+06,6.0,6.000,...,0.328567,35041.0,0.131427,18,10000000.0,0,6.0,1.000000e+07,ddos,BENIGN
148365,6.0,0.0,0,326.450665,0.000000e+00,12253,0,6.025559e+03,0.0,0.000,...,326.450665,0.0,0.000000,24,11005.0,0,0.0,0.000000e+00,ddos,BENIGN
96191,6.0,0.0,0,551.724138,0.000000e+00,7250,0,3.385003e+03,0.0,0.000,...,551.724138,0.0,0.000000,24,6286.0,0,0.0,0.000000e+00,ddos,BENIGN
9493,6.0,0.0,0,195.934362,0.000000e+00,20415,0,1.177188e+04,0.0,0.000,...,195.934362,0.0,0.000000,24,20398.0,0,0.0,0.000000e+00,ddos,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232041,0.0,1011067.0,0,0.031868,6.157050e+06,31379335,0,0.000000e+00,0.0,0.000,...,0.223077,7059238.0,0.191209,0,0.0,0,0.0,6.275823e+06,SSH,redispot
243393,0.0,0.0,0,27.539864,0.000000e+00,36311,0,0.000000e+00,0.0,0.000,...,55.079728,0.0,27.539864,0,0.0,1,0.0,0.000000e+00,elasticpot,redispot
243392,22.0,19.0,182,59.116504,2.233815e+04,202989,9,1.777505e+04,1448.0,456.875,...,98.527506,0.0,39.411003,3,37113.0,1,0.0,2.377786e+04,elasticpot,redispot
269742,56.0,297.0,6,2.196079,2.341768e+06,8651782,16,1.640721e+06,60.0,20.100,...,3.351911,682687.0,1.155831,11,7053064.0,1,0.0,9.410453e+05,SSH,redispot


In [40]:
y_test = data_final["label_dl"]
y_pred = data_final["predict_dl"]

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("Precision:", precision)
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix

c:\Users\allan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Accuracy: 0.5865458590392559
Precision: 0.9808688389738327


array([[     0,      0,      0,      0,      8,      0,      0,      0],
       [     0,    191,      0,      0,      0,   3212,      0,      0],
       [     0,     78,      0,      0,      0,    102,      0,      0],
       [     0,     11,      0,      0,      0,    164,      0,      0],
       [     0,  30310,      0,      0, 119454,  50395,      0,      0],
       [     0,     10,      0,      0,      0,     52,      0,      0],
       [     0,      3,      0,      0,      0,     46,      0,      0],
       [     0,     14,      0,      0,      0,     21,      0,      0]])